# 2.1 Load & Inspect Data
**Dataset Overview:**
- **Total Documents:** 3 PDF files (`microservices_guide.pdf`, `fastapi_best_practices.pdf`, `vector_database_fundamentals.pdf`).
- **Formats:** PDF files created using ReportLab.
- **Parsing Status:** All 3 files contain clean, extractable text using `pypdf`. No OCR required.

In [22]:
import os
from pypdf import PdfReader

docs_dir = "../data/raw_docs"
documents = []

for filename in os.listdir(docs_dir):
    if filename.endswith(".pdf"):
        filepath = os.path.join(docs_dir, filename)
        reader = PdfReader(filepath)
        full_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"
        documents.append({
            "source": filename,
            "content": full_text.strip()
        })

print(f"Successfully parsed {len(documents)} PDF documents.")

Successfully parsed 3 PDF documents.


# 2.2 Chunking Strategy
**Strategy:** Recursive Character Splitter pattern.
- **Chunk Size:** 500 characters.
- **Overlap:** 100 characters.
- **Justification:** Technical documentation contains semi-structured headers and code snippets. A 500-character chunk preserves individual functional concepts, while a 100-character overlap prevents technical terms or sentences from being cut awkwardly across boundaries.

In [23]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunked_docs = []
for doc in documents:
    raw_chunks = chunk_text(doc["content"])
    for idx, chunk in enumerate(raw_chunks):
        chunked_docs.append({
            "id": f"{doc['source']}_chunk_{idx}",
            "text": chunk,
            "source": doc["source"]
        })

print(f"Generated {len(chunked_docs)} total chunks across all documents.")

Generated 6 total chunks across all documents.


In [24]:
# 2.3 Embeddings & Vector Store Setup
import chromadb
from sentence_transformers import SentenceTransformer

# Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize persistent ChromaDB client target directory
persist_dir = "../backend/data/vector_store"
chroma_client = chromadb.PersistentClient(path=persist_dir)

# Create or reset collection
collection = chroma_client.get_or_create_collection(name="tech_docs")

# Generate embeddings and add to collection
ids = [item["id"] for item in chunked_docs]
texts = [item["text"] for item in chunked_docs]
metadatas = [{"source": item["source"]} for item in chunked_docs]
embeddings = embed_model.encode(texts).tolist()

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"Successfully stored and persisted {collection.count()} vectors into '{persist_dir}'.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully stored and persisted 6 vectors into '../backend/data/vector_store'.


In [25]:
import ollama

def retrieve_context(query, top_k=2):
    query_embedding = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    
    retrieved_chunks = results["documents"][0]
    sources = [meta["source"] for meta in results["metadatas"][0]]
    return retrieved_chunks, sources

def generate_rag_response(query):
    chunks, sources = retrieve_context(query)
    context_str = "\n\n".join([f"[Source: {src}]\n{txt}" for txt, src in zip(chunks, sources)])
    
    prompt = f"""You are a helpful assistant. Answer the user question based ONLY on the provided context below. 
If the answer cannot be found in the context, respond with "I cannot find this information in the documents."

Context:
{context_str}

Question: {query}
Answer:"""

    response = ollama.chat(
        model="qwen2.5:1.5b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"], list(set(sources))

In [26]:
import pandas as pd

test_questions = [
    "What is microservices architecture?",
    "What pattern solves data consistency in microservices?",
    "What tool handles service discovery?",
    "How do application lifespan events help performance in FastAPI?",
    "What Pydantic model is used for request validation in FastAPI?",
    "Why add CORSMiddleware?",
    "What does Cosine Similarity measure?",
    "What are the two types of chunking strategies?",
    "What step comes first in the RAG query pipeline sequence?",
    "Who won the 1998 World Cup?"  # Out of domain query test
]

eval_results = []

for q in test_questions:
    ans, sources = generate_rag_response(q)
    eval_results.append({
        "Question": q,
        "Retrieved Sources": ", ".join(sources) if sources else "None",
        "Generated Answer": ans,
        "Grounded/Correct": "Yes" if "cannot find" not in ans and len(sources) > 0 else ("N/A (Out of Domain)" if "cannot find" in ans else "No")
    })

eval_df = pd.DataFrame(eval_results)
eval_df

,Question,Retrieved Sources,Generated Answer,Grounded/Correct
0,What is microservices architecture?,microservices_guide.pdf,Microservices architecture decomposes applicat...,Yes
1,What pattern solves data consistency in micros...,microservices_guide.pdf,The Saga pattern solves data consistency in mi...,Yes
2,What tool handles service discovery?,microservices_guide.pdf,Service Discovery: Dynamic IP management requi...,Yes
3,How do application lifespan events help perfor...,"microservices_guide.pdf, fastapi_best_practice...",Application Lifespan Events in FastAPI allow r...,Yes
4,What Pydantic model is used for request valida...,fastapi_best_practices.pdf,The Pydantic model used for request validation...,Yes
5,Why add CORSMiddleware?,fastapi_best_practices.pdf,To allow requests securely from frontend appli...,Yes
6,What does Cosine Similarity measure?,vector_database_fundamentals.pdf,Cosine Similarity measures the angular distanc...,Yes
7,What are the two types of chunking strategies?,"microservices_guide.pdf, vector_database_funda...",The two types of chunking strategies mentioned...,Yes
8,What step comes first in the RAG query pipelin...,"microservices_guide.pdf, fastapi_best_practice...",I cannot find this information in the documents.,N/A (Out of Domain)
9,Who won the 1998 World Cup?,"fastapi_best_practices.pdf, vector_database_fu...",I cannot find this information in the documents.,N/A (Out of Domain)
